# Day 037 Project: Clean a Messy Sales Dataset

## What You're Building

A full cleaning pipeline applied to a realistic messy dataset: bad numeric values, whitespace in strings, missing quantities, and duplicate rows — all fixed in a reproducible sequence of steps.

**Deliverable:** You run every cell top-to-bottom. The final cell's checks pass. You have a `cleaned` DataFrame with no nulls, no extra whitespace, and no duplicates.

## Project Requirements

1. Load `MESSY_CSV` (provided) with `pd.read_csv(io.StringIO(...))`
2. Call `coerce_numeric_columns` to fix the bad `price` value
3. Call `clean_string_column` on `product` and `region`
4. Call `drop_or_fill_nulls` with `strategy='median'` to fill missing `quantity`
5. Call `deduplicate` to remove duplicate rows
6. Store the result as `cleaned` and verify with `_run_project_checks()`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result


import pandas as pd

def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result


import pandas as pd

def clean_string_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    result = df.copy()
    result[col] = result[col].str.strip().str.lower()
    return result


import pandas as pd

def deduplicate(df: pd.DataFrame, subset: list | None = None) -> pd.DataFrame:
    return df.drop_duplicates(subset=subset).reset_index(drop=True)


import pandas as pd

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for col in result.select_dtypes(include='object').columns:
        result[col] = result[col].str.strip()
    for col in result.select_dtypes(include='number').columns:
        result[col] = result[col].fillna(result[col].median())
    return result.drop_duplicates().reset_index(drop=True)


MESSY_CSV = (
    'order_id,product,price,quantity,region\n'
    '1, Widget ,25.0,10, North\n'
    '2, Gadget ,bad_price,,South\n'
    '3,Widget,25.0,20,North\n'
    '4, Widget ,25.0,20, North\n'
    '5,Doohickey,8.0,50,EAST\n'
    '5,Doohickey,8.0,50,EAST\n'
    '6,Thingamajig,200.0,8,West'
)
raw = pd.read_csv(io.StringIO(MESSY_CSV))
print(f'Raw shape: {raw.shape}')
print(raw.to_string())

## Your Cleaning Pipeline

In [ ]:
# Step 1: Fix the bad price value ('bad_price' -> NaN)
# TODO: df = coerce_numeric_columns(raw, ['price'])

# Step 2: Strip and lowercase product and region
# TODO: df = clean_string_column(df, 'product')
# TODO: df = clean_string_column(df, 'region')

# Step 3: Fill missing quantity (and coerced NaN price) with median
# TODO: df = drop_or_fill_nulls(df, strategy='median')

# Step 4: Remove duplicate rows
# TODO: cleaned = deduplicate(df)

# Step 5: Inspect the result
# TODO: print(cleaned.to_string())

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: cleaned is defined
    try:
        assert 'cleaned' in globals(), \
            "'cleaned' not defined — complete all steps and store result as 'cleaned'"
        assert isinstance(cleaned, pd.DataFrame)
        passed += 1; print('\u2705 Check 1: cleaned DataFrame defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: no nulls in cleaned
    try:
        null_total = cleaned.isnull().sum().sum()
        assert null_total == 0, \
            f'{null_total} null values remain in cleaned'
        passed += 1; print('\u2705 Check 2: no null values')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: no leading/trailing whitespace in string columns
    try:
        str_cols = cleaned.select_dtypes(include='object').columns
        for col in str_cols:
            vals = cleaned[col].dropna()
            has_ws = vals.str.startswith(' ') | vals.str.endswith(' ')
            assert not has_ws.any(), \
                f'column {col!r} still has whitespace'
        passed += 1; print('\u2705 Check 3: no whitespace in string columns')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: no duplicate rows
    try:
        assert not cleaned.duplicated().any(), \
            'duplicate rows still present'
        passed += 1; print(f'\u2705 Check 4: no duplicate rows ({len(cleaned)} rows remain)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: fewer rows than raw (cleaning removed rows)
    try:
        assert len(cleaned) < len(raw), \
            f'cleaned has same row count as raw ({len(raw)}) — dedup did not run'
        assert len(cleaned) > 0, 'cleaned is empty'
        passed += 1; print(f'\u2705 Check 5: {len(raw)} raw rows → {len(cleaned)} clean rows')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `clean_dataframe` call at the end and compare its output to your step-by-step result
- Handle the inconsistent casing in `region` — 'EAST' vs 'East' — using `clean_string_column`; what effect does it have on `deduplicate`?
- Try `strategy='mean'` vs `strategy='median'` for the null fill — which gives a more realistic replacement for `quantity`?
- Export `cleaned` to CSV with `cleaned.to_csv('cleaned_sales.csv', index=False)`
- On Day 38 you will run a full EDA on this dataset — save `cleaned` for then